# 3 · Study-topic similarity experiment

This is a side experiment with vector representations and cosine similarity. I'm using SAT study topics to practice the search concepts that also come up in my BioCosmos work.

I started with six keyword-count features so I can inspect what each vector represents. The search ranks topics by cosine similarity and joins the results to their titles and descriptions in DuckDB.

This isn't a pretrained embedding model, and it doesn't affect the score predictions. A later experiment could compare these rankings with a text embedding model, but I haven't implemented that comparison yet.

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project folder or in notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "satlab").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
from satlab.database import connect, seed_demo, search_topics
from satlab.similarity import FEATURES, embed, cosine_similarity

db = connect(ROOT / "data" / "learning.duckdb")
seed_demo(db)
display(db.execute("SELECT * FROM topics ORDER BY topic_id").df())


## 1. Encoding a query

I assign each coordinate to a vocabulary: algebra, geometry, probability, grammar, evidence, or data. A coordinate counts matching words from its vocabulary.

Stored topic vectors and query vectors use the same encoder, dimensions, and coordinate order. A zero vector has no direction, so the implementation rejects it when calculating cosine similarity.

The example query focuses on probability and guessing. Queries such as `linear equation slope` or `grammar comma sentence` would emphasize different coordinates.

In [ ]:
query = "probability guessing distribution"
query_vector = embed(query)
display(pd.Series(query_vector, index=FEATURES, name="Query vector"))


## 2. Deriving cosine similarity

I compare vector directions using

$$\cos(\theta)=\frac{a\cdot b}{\|a\|_2\|b\|_2}.$$

The dot product is $\sum_i a_i b_i$, and $\|a\|_2=\sqrt{\sum_i a_i^2}$.
For unit-normalized vectors, the denominator is 1, so cosine similarity equals the dot product.
Cauchy–Schwarz gives $|a\cdot b|\leq\|a\|\|b\|$, placing cosine in $[-1,1]$ for nonzero real vectors.

Multiplying a vector by a positive scalar preserves its direction and therefore its cosine similarity. The code below checks this property.
My nonnegative keyword vectors produce similarities in $[0,1]$. These values measure directional agreement, not a probability of answer correctness or study benefit.

In [ ]:
a, b = [1., 0., 2.], [2., 0., 1.]
manual = sum(x*y for x,y in zip(a,b)) / (sum(x*x for x in a)**.5 * sum(y*y for y in b)**.5)
print("Manual:", manual, "Function:", cosine_similarity(a,b))
assert abs(cosine_similarity(a,b) - cosine_similarity([10*x for x in a],b)) < 1e-12


## 3. Searching in DuckDB

I store `topic_id`, encoder version, and vector in `topic_embeddings`, while `topics` holds titles and descriptions.
The query computes `list_cosine_similarity`, sorts the results, and joins the readable metadata using `topic_id`.

This small dataset uses an exact scan. The assertion below compares the top SQL result with the Python cosine implementation to check that both calculations agree.

In [ ]:
results = search_topics(db, query_vector, limit=4)
display(results)
best_id = results.iloc[0]["topic_id"]
best_vector = db.execute("SELECT embedding FROM topic_embeddings WHERE topic_id = ?", [best_id]).fetchone()[0]
assert abs(results.iloc[0]["similarity"] - cosine_similarity(query_vector, best_vector)) < 1e-10


## 4. Looking at the vectors

I plot the stored vectors as a heatmap to show which feature coordinates contribute to each topic's representation. This makes the keyword baseline's behavior visible.

A pretrained text encoder could provide a richer representation, although individual dimensions may be less interpretable. Whether that improves retrieval would need to be evaluated against relevance judgments.

In [ ]:
vectors = db.execute("""
    SELECT t.title, e.embedding FROM topics t JOIN topic_embeddings e USING (topic_id)
    WHERE embedding_model='keyword-count-v1' ORDER BY t.topic_id
""").fetchall()
matrix = np.array([row[1] for row in vectors])
fig, ax = plt.subplots(figsize=(9, 4))
image = ax.imshow(matrix, cmap="YlGn")
ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=25, ha="right")
ax.set_yticks(range(len(vectors)), [row[0] for row in vectors])
ax.set_title("Inspect the teaching vectors")
fig.colorbar(image, ax=ax, label="Keyword count")
plt.tight_layout()
plt.show()

artifact = ROOT / "artifacts"
artifact.mkdir(exist_ok=True)
(artifact / "topic_search.json").write_text(results.to_json(orient="records", indent=2))
print("Exported joined search results for a possible future web interface.")
db.close()


## 5. Possible next steps

I would extend this retrieval example by adding more SAT study resources and comparing the rankings with relevance judgments. A larger schema could separate `study_resources`, `resource_embeddings`, and `search_runs`, with shared IDs connecting the records.

I'd keep track of the encoder version and vector size so I don't accidentally compare incompatible vectors. A comparison with a pretrained text encoder would help me test whether a more complex representation actually improves the rankings.

I'd evaluate the search separately from the predictor. A high similarity score doesn't tell me whether a study resource will improve someone's SAT score.

Reference: [DuckDB list functions](https://duckdb.org/docs/current/sql/functions/list).